# Notebook Overview — Combine and Split Metadata

## Purpose

This notebook combines six preprocessed dataset metadata CSV files and creates balanced **train** and **test** split CSV files.

---

## Inputs

The notebook expects the following six preprocessed metadata CSV files to be available from the GitHub repository and loaded into the local runtime:

* `metadata/preprocessed/imgn_preprocessed_metadata.csv`
* `metadata/preprocessed/coco_preprocessed_metadata.csv`
* `metadata/preprocessed/diff_preprocessed_metadata.csv`
* `metadata/preprocessed/sdxl_preprocessed_metadata.csv`
* `metadata/preprocessed/mj_preprocessed_metadata.csv`
* `metadata/preprocessed/open_preprocessed_metadata.csv`

---

## Assumptions

* Each input CSV contains a `filename` column

* Each input CSV contains **3000 rows**

* The datasets correspond to:

  | Dataset            | Class |
  | ------------------ | ----- |
  | ImageNet_1K_256    | real  |
  | MS_COCO_2017       | real  |
  | OpenImages         | real  |
  | DiffusionDB        | ai    |
  | SDXL_Generated_10K | ai    |
  | Midjourney         | ai    |

* This notebook does **not** move, copy, unzip, or preprocess images

* This notebook operates entirely on metadata CSV files

* Input CSV files are obtained from GitHub and processed in the local runtime environment

---

## What This Notebook Does

### Cell 1 — Environment Setup

* Import project configuration
* Define local runtime paths (GitHub-based overrides)
* Verify required input files are available

---

### Cell 2 — Dataset Configuration

* Define dataset metadata:

  * CSV filenames
  * source dataset names
  * class labels

---

### Cell 3 — Combine Metadata

* Load all six input CSV files
* Add:

  * `source_dataset`
  * `class_label`
  * `image_path`
* Combine into a single dataframe
* Save:

  * `combined_metadata.csv`

---

### Cell 4 — Train/Test Split, Save, and Validation

For each dataset independently:

1. Shuffle rows
2. Split into exact counts:

   * **train = 2400**
   * **test = 600**

Then:

* Combine all training splits
* Combine all test splits
* Shuffle each subset independently

Save:

* `train_metadata.csv`
* `test_metadata.csv`

Finally:

* Verify output files exist
* Display dataset shapes for:

  * `combined_metadata.csv`
  * `train_metadata.csv`
  * `test_metadata.csv`

---

## Outputs

The following files are written to the local runtime (and may optionally be saved or downloaded):

* `metadata/splits/combined_metadata.csv`
* `metadata/splits/train_metadata.csv`
* `metadata/splits/test_metadata.csv`

---

## Expected Sizes

* `combined_metadata.csv` → **18,000 rows**
* `train_metadata.csv` → **14,400 rows**
* `test_metadata.csv` → **3,600 rows**

---

## Notes

* The split is **balanced by dataset and class label**
* A fixed random seed ensures **reproducibility**
* The training set is intended for **k-fold cross-validation** in later notebooks
* Images remain in their original storage locations; metadata defines dataset structure
* This notebook represents the first stage of the pipeline that operates entirely on structured metadata rather than raw images

---

**Next step:** The following cell performs environment setup and verifies required inputs.


In [3]:
# ============================================
# Cell 0: Startup (Environment + Verification)
# ============================================

import os
import sys
from pathlib import Path

# -------------------------------------------------
# Clone repo into Colab runtime (if not already present)
# -------------------------------------------------
REPO_URL = "https://github.com/pgailinas/dip-ai-image-detection.git"
REPO_DIR = Path("/content/dip-ai-image-detection")

if not REPO_DIR.exists():
    print("Cloning repository...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

# -------------------------------------------------
# Make src/ importable
# -------------------------------------------------
SRC_DIR = REPO_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# -------------------------------------------------
# Import shared project configuration
# -------------------------------------------------
from project_config import (
    DATASET_CODES,
    PREPROCESSED_METADATA_COLUMNS,
    AI_LABEL,
    REAL_LABEL,
    TRAIN_RATIO,
    TEST_RATIO,
    TRAIN_SUBSET,
    TEST_SUBSET,
    TRAIN_PER_SOURCE,
    TEST_PER_SOURCE,
    RANDOM_SEED,
    COMBINED_METADATA_FILENAME,
    TRAIN_METADATA_FILENAME,
    TEST_METADATA_FILENAME,
)

# -------------------------------------------------
# Notebook 03 path overrides (minimal)
# -------------------------------------------------
METADATA_ROOT = REPO_DIR / "metadata"
INPUT_CSV_DIR = METADATA_ROOT / "preprocessed"
OUTPUT_DIR = METADATA_ROOT / "splits"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------
# Verify required input files
# -------------------------------------------------
INPUT_FILES = [
    "imgn_preprocessed_metadata.csv",
    "coco_preprocessed_metadata.csv",
    "diff_preprocessed_metadata.csv",
    "sdxl_preprocessed_metadata.csv",
    "mj_preprocessed_metadata.csv",
    "open_preprocessed_metadata.csv",
]

print("Verifying required metadata CSV files...\n")

missing_files = []

for filename in INPUT_FILES:
    if not (INPUT_CSV_DIR / filename).exists():
        missing_files.append(filename)

if missing_files:
    raise FileNotFoundError(
        f"Missing required files in metadata/preprocessed/: {missing_files}"
    )

print("All required metadata CSV files are present.")
print(f"Input directory:  {INPUT_CSV_DIR}")
print(f"Output directory: {OUTPUT_DIR}")



Verifying required metadata CSV files...

All required metadata CSV files are present.
Input directory:  /content/dip-ai-image-detection/metadata/preprocessed
Output directory: /content/dip-ai-image-detection/metadata/splits


In [4]:
# =========================
# Cell 1: Define Paths
# =========================

from pathlib import Path
import pandas as pd

print("Paths defined.")
print(f"INPUT_CSV_DIR = {INPUT_CSV_DIR}")
print(f"OUTPUT_DIR    = {OUTPUT_DIR}")



Paths defined.
INPUT_CSV_DIR = /content/dip-ai-image-detection/metadata/preprocessed
OUTPUT_DIR    = /content/dip-ai-image-detection/metadata/splits


In [5]:
# =========================
# Cell 2: Dataset configuration
# =========================

DATASET_CONFIG = {
    "ImageNet_1K_256": {
        "csv": "imgn_preprocessed_metadata.csv",
        "class_label": REAL_LABEL,
    },
    "MS_COCO_2017": {
        "csv": "coco_preprocessed_metadata.csv",
        "class_label": REAL_LABEL,
    },
    "OpenImages": {
        "csv": "open_preprocessed_metadata.csv",
        "class_label": REAL_LABEL,
    },
    "DiffusionDB": {
        "csv": "diff_preprocessed_metadata.csv",
        "class_label": AI_LABEL,
    },
    "SDXL_Generated_10K": {
        "csv": "sdxl_preprocessed_metadata.csv",
        "class_label": AI_LABEL,
    },
    "Midjourney": {
        "csv": "mj_preprocessed_metadata.csv",
        "class_label": AI_LABEL,
    },
}

print("Dataset configurations defined.")


Dataset configurations defined.


In [6]:
# =========================
# Cell 3: Load and combine metadata
# =========================
dfs = []

BASE_COLUMNS = ["filename"]

for dataset_name, cfg in DATASET_CONFIG.items():
    csv_path = INPUT_CSV_DIR / cfg["csv"]

    if not csv_path.exists():
        raise FileNotFoundError(f"Missing input CSV: {csv_path}")

    df = pd.read_csv(csv_path)

    if "filename" not in df.columns:
        raise ValueError(f"'filename' column not found in {csv_path}")

    # Keep only required columns
    df = df[BASE_COLUMNS].copy()

    # Add standardized fields
    df["source_dataset"] = dataset_name
    df["class_label"] = cfg["class_label"]

    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)

print("Combined dataset shape:", combined_df.shape)

print("\nCounts by source_dataset:")
print(combined_df["source_dataset"].value_counts())

print("\nCounts by class_label:")
print(combined_df["class_label"].value_counts())

print("\nColumns in combined dataset:")
print(combined_df.columns.tolist())

combined_csv_path = OUTPUT_DIR / COMBINED_METADATA_FILENAME
combined_df.to_csv(combined_csv_path, index=False)

print(f"\nSaved combined metadata to: {combined_csv_path}")


Combined dataset shape: (18000, 3)

Counts by source_dataset:
source_dataset
ImageNet_1K_256       3000
MS_COCO_2017          3000
OpenImages            3000
DiffusionDB           3000
SDXL_Generated_10K    3000
Midjourney            3000
Name: count, dtype: int64

Counts by class_label:
class_label
rl    9000
ai    9000
Name: count, dtype: int64

Columns in combined dataset:
['filename', 'source_dataset', 'class_label']

Saved combined metadata to: /content/dip-ai-image-detection/metadata/splits/combined_metadata.csv


In [9]:
# =========================
# Cell 4: Split + Save + Validate Metadata
# =========================

split_dfs = []

# -------------------------------------------------
# Split each dataset into train/test
# -------------------------------------------------
for dataset_name, group_df in combined_df.groupby("source_dataset"):
    # Shuffle each dataset independently
    group_df = group_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

    # Exact-count split
    train_df = group_df.iloc[:TRAIN_PER_SOURCE].copy()
    test_df = group_df.iloc[
        TRAIN_PER_SOURCE:TRAIN_PER_SOURCE + TEST_PER_SOURCE
    ].copy()

    train_df["subset"] = TRAIN_SUBSET
    test_df["subset"] = TEST_SUBSET

    split_dfs.extend([train_df, test_df])

# -------------------------------------------------
# Combine all splits
# -------------------------------------------------
split_df = pd.concat(split_dfs, ignore_index=True)

train_metadata = split_df[split_df["subset"] == TRAIN_SUBSET].copy()
test_metadata = split_df[split_df["subset"] == TEST_SUBSET].copy()

# Final shuffle of each subset independently
train_metadata = train_metadata.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
test_metadata = test_metadata.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

# -------------------------------------------------
# Save outputs
# -------------------------------------------------
train_csv_path = OUTPUT_DIR / TRAIN_METADATA_FILENAME
test_csv_path = OUTPUT_DIR / TEST_METADATA_FILENAME

train_metadata.to_csv(train_csv_path, index=False)
test_metadata.to_csv(test_csv_path, index=False)

# -------------------------------------------------
# Reporting
# -------------------------------------------------
print("Train shape:", train_metadata.shape)
print("Test shape:", test_metadata.shape)

print("\nCounts by subset:")
print(pd.Series({
    TRAIN_SUBSET: len(train_metadata),
    TEST_SUBSET: len(test_metadata),
}))

print("\nCounts by subset and source_dataset:")
print(split_df.groupby(["subset", "source_dataset"]).size())

print("\nCounts by subset and class_label:")
print(split_df.groupby(["subset", "class_label"]).size())

print(f"\nSaved train metadata to: {train_csv_path}")
print(f"Saved test metadata to: {test_csv_path}")

# -------------------------------------------------
# Validation of output CSV files
# -------------------------------------------------
print("\nValidation of saved CSV files:")

for csv_name in [
    COMBINED_METADATA_FILENAME,
    TRAIN_METADATA_FILENAME,
    TEST_METADATA_FILENAME,
]:
    csv_path = OUTPUT_DIR / csv_name
    if not csv_path.exists():
        print(f"Missing: {csv_path}")
    else:
        df = pd.read_csv(csv_path)
        print(f"{csv_name}: {df.shape}")


Train shape: (14400, 4)
Test shape: (3600, 4)

Counts by subset:
train    14400
test      3600
dtype: int64

Counts by subset and source_dataset:
subset  source_dataset    
test    DiffusionDB            600
        ImageNet_1K_256        600
        MS_COCO_2017           600
        Midjourney             600
        OpenImages             600
        SDXL_Generated_10K     600
train   DiffusionDB           2400
        ImageNet_1K_256       2400
        MS_COCO_2017          2400
        Midjourney            2400
        OpenImages            2400
        SDXL_Generated_10K    2400
dtype: int64

Counts by subset and class_label:
subset  class_label
test    ai             1800
        rl             1800
train   ai             7200
        rl             7200
dtype: int64

Saved train metadata to: /content/dip-ai-image-detection/metadata/splits/train_metadata.csv
Saved test metadata to: /content/dip-ai-image-detection/metadata/splits/test_metadata.csv

Validation of saved CSV files: